In [4]:
import cv2
import numpy as np
import glob
import os
import torch
from ultralytics import FastSAM


output_folder = "AI_results"
os.makedirs(output_folder, exist_ok=True)


model = FastSAM('FastSAM-s.pt')


image_files = sorted(glob.glob("*.jpg"))

if not image_files:
    print("")

for image_path in image_files:
    base_name = os.path.splitext(os.path.basename(image_path))[0]


    image = cv2.imread(image_path)
    orig_h, orig_w, _ = image.shape
    new_w = 800
    new_h = int(orig_h * (new_w / orig_w))
    resized_img = cv2.resize(image, (new_w, new_h))
    output = resized_img.copy()

    results = model(resized_img, device=0, retina_masks=True, imgsz=640, conf=0.4, iou=0.9, verbose=False)
    res = results[0]

    binary_mask = np.zeros((new_h, new_w), dtype=np.uint8)
    color_mask = np.zeros_like(resized_img)

    count = 0
    if res.masks is not None:
        masks = res.masks.data.cpu().numpy()
        for m in masks:
            m_resized = cv2.resize(m.astype(np.uint8), (new_w, new_h))
            area = np.sum(m_resized)


            y_indices, x_indices = np.where(m_resized == 1)
            if len(x_indices) > 0:
                touches_border = (
                    (np.min(x_indices) <= 3) or (np.min(y_indices) <= 3) or
                    (np.max(x_indices) >= new_w - 3) or (np.max(y_indices) >= new_h - 3)
                )


                if 50 < area < 4000 and not touches_border:
                    count += 1
                    binary_mask[m_resized == 1] = 255
                    color_mask[m_resized == 1] = (0, 255, 0)


                    contours, _ = cv2.findContours(m_resized, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    cv2.drawContours(output, contours, -1, (0, 255, 0), 2)

                    cx = int(np.mean(x_indices))
                    cy = int(np.mean(y_indices))
                    cv2.putText(output, str(count), (cx - 6, cy + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)


    final_overlay = cv2.addWeighted(output, 0.65, color_mask, 0.35, 0)
    cv2.putText(final_overlay, f"AI Total Screws: {count}", (25, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 0, 0), 3)

    cv2.imwrite(os.path.join(output_folder, f"{base_name}_ai_mask.png"), binary_mask)
    cv2.imwrite(os.path.join(output_folder, f"{base_name}_ai_counted.png"), final_overlay)


    torch.cuda.empty_cache()

    print(f"AI Saved results for {image_path} -> Total Screws: {count}")


AI Saved results for 20240713_193135.jpg -> Total Screws: 48
AI Saved results for 20240713_193650.jpg -> Total Screws: 16
AI Saved results for 20240713_193659.jpg -> Total Screws: 12
AI Saved results for 20240713_193831.jpg -> Total Screws: 151
AI Saved results for 20240713_193839.jpg -> Total Screws: 115
AI Saved results for 20240713_193907.jpg -> Total Screws: 95
AI Saved results for 20240713_194200.jpg -> Total Screws: 80
AI Saved results for 20240713_194206.jpg -> Total Screws: 72
AI Saved results for 20240713_194215.jpg -> Total Screws: 83
AI Saved results for 20240713_194232.jpg -> Total Screws: 80
AI Saved results for 20240713_194256.jpg -> Total Screws: 67
AI Saved results for 20240713_194316.jpg -> Total Screws: 72
AI Saved results for 20240713_194541.jpg -> Total Screws: 54
AI Saved results for 20240713_194551.jpg -> Total Screws: 40
AI Saved results for 20240713_194606.jpg -> Total Screws: 43
AI Saved results for 20240713_194621.jpg -> Total Screws: 80
AI Saved results for 2

In [7]:
import cv2
import numpy as np
import glob
import os
import torch
from ultralytics import FastSAM

output_folder = "AI_results_HD"
os.makedirs(output_folder, exist_ok=True)

model = FastSAM('FastSAM-s.pt')
image_files = sorted(glob.glob("*.jpg"))

for image_path in image_files:
    base_name = os.path.splitext(os.path.basename(image_path))[0]

    orig_image = cv2.imread(image_path)
    orig_h, orig_w, _ = orig_image.shape
    output = orig_image.copy()

    proc_w = 1024
    proc_h = int(orig_h * (proc_w / orig_w))
    small_img = cv2.resize(orig_image, (proc_w, proc_h))

    # conf=0.25 रखने से पास-पास रखे पेंच भी आसानी से पकड़े जाएंगे
    results = model(small_img, device=0, retina_masks=True, imgsz=1024, conf=0.25, iou=0.6, verbose=False)
    res = results[0]

    binary_mask = np.zeros((orig_h, orig_w), dtype=np.uint8)
    color_mask = np.zeros_like(orig_image)

    count = 0
    if res.masks is not None:
        masks = res.masks.data.cpu().numpy()
        areas = [np.sum(m) for m in masks]
        sorted_indices = np.argsort(areas)[::-1]

        for idx in sorted_indices:
            m = masks[idx]
            m_full = cv2.resize(m.astype(np.uint8), (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
            m_small = cv2.resize(m.astype(np.uint8), (proc_w, proc_h))
            small_area = np.sum(m_small)

            y_indices, x_indices = np.where(m_full == 1)
            if len(x_indices) > 0:
                touches_border = (
                    (np.min(x_indices) <= 10) or (np.min(y_indices) <= 10) or
                    (np.max(x_indices) >= orig_w - 10) or (np.max(y_indices) >= orig_h - 10)
                )

                overlap = np.sum(np.logical_and(binary_mask == 255, m_full == 1))
                total_pixels = np.sum(m_full)

                if total_pixels > 0 and 100 < small_area < 6000 and not touches_border and (overlap / total_pixels < 0.25):
                    count += 1
                    binary_mask[m_full == 1] = 255
                    color_mask[m_full == 1] = (0, 255, 0)

                    contours, _ = cv2.findContours(m_full, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    cv2.drawContours(output, contours, -1, (0, 255, 0), 4)

                    cx = int(np.mean(x_indices))
                    cy = int(np.mean(y_indices))
                    cv2.putText(output, str(count), (cx - 15, cy + 15), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4)

    final_overlay = cv2.addWeighted(output, 0.7, color_mask, 0.3, 0)
    cv2.putText(final_overlay, f"AI Total Screws: {count}", (60, 120), cv2.FONT_HERSHEY_SIMPLEX, 3.0, (255, 0, 0), 8)

    cv2.imwrite(os.path.join(output_folder, f"{base_name}_ai_mask.png"), binary_mask)
    cv2.imwrite(os.path.join(output_folder, f"{base_name}_ai_counted.png"), final_overlay)

    torch.cuda.empty_cache()
    print(f"HD Result for {image_path} -> Total Screws: {count}")


HD Result for 20240713_193135.jpg -> Total Screws: 27
HD Result for 20240713_193650.jpg -> Total Screws: 5
HD Result for 20240713_193659.jpg -> Total Screws: 7
HD Result for 20240713_193831.jpg -> Total Screws: 141
HD Result for 20240713_193839.jpg -> Total Screws: 123
HD Result for 20240713_193907.jpg -> Total Screws: 118
HD Result for 20240713_194200.jpg -> Total Screws: 68
HD Result for 20240713_194206.jpg -> Total Screws: 66
HD Result for 20240713_194215.jpg -> Total Screws: 73
HD Result for 20240713_194232.jpg -> Total Screws: 66
HD Result for 20240713_194256.jpg -> Total Screws: 55
HD Result for 20240713_194316.jpg -> Total Screws: 65
HD Result for 20240713_194541.jpg -> Total Screws: 239
HD Result for 20240713_194551.jpg -> Total Screws: 197
HD Result for 20240713_194606.jpg -> Total Screws: 277
HD Result for 20240713_194621.jpg -> Total Screws: 281
HD Result for 20240713_194630.jpg -> Total Screws: 281


In [8]:
import cv2
import numpy as np
import glob
import os
import torch
from ultralytics import FastSAM

output_folder = "AI_results_HD"
os.makedirs(output_folder, exist_ok=True)

model = FastSAM('FastSAM-s.pt')
image_files = sorted(glob.glob("*.jpg"))

for image_path in image_files:
    base_name = os.path.splitext(os.path.basename(image_path))[0]

    orig_image = cv2.imread(image_path)
    orig_h, orig_w, _ = orig_image.shape
    output = orig_image.copy()

    proc_w = 1024
    proc_h = int(orig_h * (proc_w / orig_w))
    small_img = cv2.resize(orig_image, (proc_w, proc_h))


    results = model(small_img, device=0, retina_masks=True, imgsz=1024, conf=0.35, iou=0.45, verbose=False)
    res = results[0]

    binary_mask = np.zeros((orig_h, orig_w), dtype=np.uint8)
    color_mask = np.zeros_like(orig_image)

    count = 0
    if res.masks is not None:
        masks = res.masks.data.cpu().numpy()
        areas = [np.sum(m) for m in masks]

        sorted_indices = np.argsort(areas)

        for idx in sorted_indices:
            m = masks[idx]
            m_full = cv2.resize(m.astype(np.uint8), (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
            m_small = cv2.resize(m.astype(np.uint8), (proc_w, proc_h))
            small_area = np.sum(m_small)

            y_indices, x_indices = np.where(m_full == 1)
            if len(x_indices) > 0:
                touches_border = (
                    (np.min(x_indices) <= 10) or (np.min(y_indices) <= 10) or
                    (np.max(x_indices) >= orig_w - 10) or (np.max(y_indices) >= orig_h - 10)
                )

                overlap = np.sum(np.logical_and(binary_mask == 255, m_full == 1))
                total_pixels = np.sum(m_full)


                if total_pixels > 0 and 150 < small_area < 2800 and not touches_border and (overlap / total_pixels < 0.2):
                    count += 1
                    binary_mask[m_full == 1] = 255
                    color_mask[m_full == 1] = (0, 255, 0)

                    contours, _ = cv2.findContours(m_full, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    cv2.drawContours(output, contours, -1, (0, 255, 0), 4)

                    cx = int(np.mean(x_indices))
                    cy = int(np.mean(y_indices))


                    label = str(count)
                    cv2.putText(output, label, (cx - 20, cy + 15), cv2.FONT_HERSHEY_SIMPLEX, 1.8, (0, 0, 0), 8)
                    cv2.putText(output, label, (cx - 20, cy + 15), cv2.FONT_HERSHEY_SIMPLEX, 1.8, (0, 255, 255), 4)

    final_overlay = cv2.addWeighted(output, 0.7, color_mask, 0.3, 0)
    cv2.putText(final_overlay, f"AI Total Screws: {count}", (60, 130), cv2.FONT_HERSHEY_SIMPLEX, 3.2, (0, 0, 0), 12)
    cv2.putText(final_overlay, f"AI Total Screws: {count}", (60, 130), cv2.FONT_HERSHEY_SIMPLEX, 3.2, (255, 255, 0), 6)

    cv2.imwrite(os.path.join(output_folder, f"{base_name}_ai_mask.png"), binary_mask)
    cv2.imwrite(os.path.join(output_folder, f"{base_name}_ai_counted.png"), final_overlay)

    torch.cuda.empty_cache()
    print(f"HD Result for {image_path} -> Total Screws: {count}")


HD Result for 20240713_193135.jpg -> Total Screws: 40
HD Result for 20240713_193650.jpg -> Total Screws: 8
HD Result for 20240713_193659.jpg -> Total Screws: 10
HD Result for 20240713_193831.jpg -> Total Screws: 149
HD Result for 20240713_193839.jpg -> Total Screws: 138
HD Result for 20240713_193907.jpg -> Total Screws: 129
HD Result for 20240713_194200.jpg -> Total Screws: 78
HD Result for 20240713_194206.jpg -> Total Screws: 79
HD Result for 20240713_194215.jpg -> Total Screws: 80
HD Result for 20240713_194232.jpg -> Total Screws: 79
HD Result for 20240713_194256.jpg -> Total Screws: 68
HD Result for 20240713_194316.jpg -> Total Screws: 73
HD Result for 20240713_194541.jpg -> Total Screws: 214
HD Result for 20240713_194551.jpg -> Total Screws: 167
HD Result for 20240713_194606.jpg -> Total Screws: 281
HD Result for 20240713_194621.jpg -> Total Screws: 294
HD Result for 20240713_194630.jpg -> Total Screws: 292
